# BGF Demo Notebook

CPU-only end-to-end demonstration of the Behavioral Grounding Framework.
Runs in under 60 seconds on a laptop.

Steps covered:
1. Population synthesis (10 agents)
2. 5-round simulation (MockPolicy, no GPU)
3. Policy intervention (redistribute_top_pct, p=0.20)
4. TSTR synthetic utility benchmark

In [1]:
# Cell 1 — Setup
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# Also try parent in case notebook is run from notebooks/ subdirectory
parent = PROJECT_ROOT.parent
if (parent / 'agents').exists() and str(parent) not in sys.path:
    sys.path.insert(0, str(parent))
    PROJECT_ROOT = parent

print(f'Project root: {PROJECT_ROOT}')

Project root: /mnt/sdb1/workspace/lucastourinho/SyntheticSocieties


In [2]:
# Cell 2 — Population synthesis
import numpy as np

from agents.agent import Agent
from agents.memory import MemoryBuffer
from agents.profile import AgentProfile
from agents.state import AgentState
from decision.mock_policy import MockPolicy

N_AGENTS = 10
# Explicitly alternate trust values above/below 0.5 to ensure both TSTR label classes.
trust_values = [0.25, 0.75, 0.30, 0.80, 0.20, 0.70, 0.35, 0.65, 0.40, 0.60]

agents = []
for i in range(N_AGENTS):
    profile = AgentProfile(
        agent_id=f'agent_{i:03d}',
        age=30 + i * 4,
        income=float(30 + i * 8),
        education='secondary',
        occupation='worker',
        location='EU',
        political_preference='centrist',
        risk_tolerance=0.5,
        social_class='middle',
        trust_people=trust_values[i],
    )
    state = AgentState(wealth=30.0 + i * 8.0)
    memory = MemoryBuffer(max_items=10)
    policy = MockPolicy()
    agents.append(Agent(profile=profile, state=state, memory=memory, policy=policy))

print(f'Created {len(agents)} agents')
print(f'Wealth range: [{agents[0].state.wealth:.1f}, {agents[-1].state.wealth:.1f}]')

Created 10 agents
Wealth range: [30.0, 102.0]


In [3]:
# Cell 3 — Run 5-round simulation
import tempfile, os

from bgf_logging.event_logger import EventLogger
from environment.institutions import InstitutionManager
from environment.world import World
from environment.world_state import WorldState
from metrics.inequality import gini_coefficient
from simulation.kernel import SimulationKernel

world_state = WorldState(round_id=0)
institution_manager = InstitutionManager()
world = World(state=world_state, institution_manager=institution_manager, network_manager=None)

with tempfile.NamedTemporaryFile(suffix='.jsonl', delete=False) as tf:
    tmp_log = tf.name

logger = EventLogger(output_path=tmp_log, overwrite=True)
kernel = SimulationKernel(agents=agents, world=world, logger=logger)
kernel.run(5)

wealths = [a.state.wealth for a in agents]
gini_pre = gini_coefficient(wealths)

actions = [a.state.last_action for a in agents if a.state.last_action]
coop_rate = sum(1 for a in actions if a == 'cooperate') / len(actions) if actions else 0.0

os.unlink(tmp_log)

print(f'After 5 rounds:')
print(f'  Gini coefficient : {gini_pre:.4f}')
print(f'  Cooperation rate : {coop_rate:.4f}')

After 5 rounds:
  Gini coefficient : 0.0921
  Cooperation rate : 0.0000


In [4]:
# Cell 4 — Policy intervention: redistribute_top_pct, p=0.20
from environment.policy_intervention import InterventionEngine, PolicyIntervention

trigger = world.state.round_id
intervention = PolicyIntervention(
    trigger_round=trigger,
    rule='redistribute_top_pct',
    parameter=0.20,
    label='20pct-wealth-tax',
)
engine = InterventionEngine([intervention])
events = engine.apply(round_id=world.state.round_id, agents=agents)

gini_post = gini_coefficient([a.state.wealth for a in agents])
delta_gini = gini_post - gini_pre

print(f'Policy: redistribute_top_pct (p=0.20)')
print(f'  Gini before : {gini_pre:.4f}')
print(f'  Gini after  : {gini_post:.4f}')
direction = 'more equal' if delta_gini < 0 else 'no change / increase'
print(f'  DeltaGini   : {delta_gini:+.4f}  ({direction})')
print(f'  Events fired: {len(events)}')

Policy: redistribute_top_pct (p=0.20)
  Gini before : 0.0921
  Gini after  : 0.0554
  DeltaGini   : -0.0367  (more equal)
  Events fired: 1


In [5]:
# Cell 5 — TSTR synthetic utility benchmark
from metrics.synthetic_utility import build_synthetic_dataset, tstr_benchmark, utility_report

# build_synthetic_dataset expects profiles (AgentProfile), not Agent objects
profiles = [a.profile for a in agents]
synthetic_X, synthetic_y = build_synthetic_dataset(profiles)

# Noise-perturbed real dataset (same profiles, slight perturbation)
rng2 = np.random.default_rng(99)
real_X = (synthetic_X + rng2.normal(0, 0.05, synthetic_X.shape)).clip(0, 1).astype('float32')
real_y = synthetic_y.copy()

result = tstr_benchmark(synthetic_X, synthetic_y, real_X, real_y)
report = utility_report(result)

  Synthetic Data Utility Benchmark (TSTR vs TRTR)

  Dataset sizes
    Synthetic train:      10
    Real train:            7
    Real test:             3

  Label balance (fraction positive)
    Synthetic:        0.5000
    Real:             0.5000

  Condition              Accuracy    AUC-ROC
  ------------------------------------------
  TRTR (train real)        0.3333     0.5000
  TSTR (train synth)       0.6667     1.0000
  ------------------------------------------
  Utility gap             -0.3334    -0.5000

  Utility threshold (≤ 0.05):  PASS ✓

